# SCALE x ODYSSEY -- 03: Evaluation & Visualization

Evaluate model, generate confusion matrix, and visualize Grad-CAM.

In [ ]:
import sys
sys.path.insert(0, '../src')

from utils import load_config, load_checkpoint
from dataset import AstroDataset, get_loaders
from model import AstroClassifier
from evaluate import evaluate_standard, plot_confusion_matrix, plot_per_class_metrics
from gradcam import visualize_predictions
import torch
import matplotlib.pyplot as plt

config = load_config('../configs/config.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Load Model

In [ ]:
model = AstroClassifier(config['model']['num_classes'], config['model']['backbone'],
                        pretrained=False, dropout=config['model']['dropout']).to(device)
load_checkpoint('checkpoints/best_model.pth', model, device)
print('Model loaded successfully')

## Standard Evaluation

In [ ]:
_, _, test_loader = get_loaders(config['data']['processed_dir'],
                                 batch_size=config['training']['batch_size'],
                                 num_workers=config['data']['num_workers'])
results = evaluate_standard(model, test_loader, device)
print(f"Accuracy: {results['accuracy']:.4f}")
print(f"Macro F1: {results['macro_f1']:.4f}")
print('\nPer-Class F1:')
for cls, f1 in results['per_class_f1'].items():
    print(f'  {cls}: {f1:.4f}')
print(f"\n{results['classification_report']}")

## Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(results['labels'], results['predictions'])
plot_confusion_matrix(cm, 'confusion_matrix_notebook.png')
from IPython.display import Image as IPImage, display
display(IPImage(filename='confusion_matrix_notebook.png'))

## Grad-CAM Visualizations

In [ ]:
test_dataset = AstroDataset(config['data']['processed_dir'], 'test', config['data']['image_size'])
visualize_predictions(model=model, dataset=test_dataset, device=device,
                      output_dir='../results/gradcam_notebook', num_samples=6,
                      image_size=config['data']['image_size'])

## View Summary Grid

In [ ]:
from IPython.display import Image as IPImage, display
display(IPImage(filename='../results/gradcam_notebook/_summary_grid.png'))